# Budget follow-up: model comparison and fresh Beer Games

**Open this notebook to review the follow-up already run by the assistant.** No API calls occur with the default `RUN_API = False`.

The fixed development comparison used GPT-5.4 nano, GPT-5.4 mini and GPT-5.6 Luna with original/clarified instructions. Selection used factual comprehension, valid replies and price; no H1/H2 results were used. The selected Luna model did not hit a 96-token cap, so the fresh calibration uses **32 / 240 / 480 tokens**. Games start only if that calibration passes. The follow-up plans **two chains per condition, four role agents per chain**.

This is a feasibility pilot. Earlier results remain separate; more trials or a more expensive model cannot guarantee statistical significance.

**Completed results:** 4 chains, 16 role agents, 576 decisions; no zero orders; orders ranged from 4 to 12. Short-chain costs were **852 and 240**, versus **156 and 156** with adequate tokens. These two pairs support the predicted direction of H2 descriptively, without statistical confirmation. **H1 remains unidentified:** adequate agents copied incoming demand, so the trajectories do not identify a separate supply-line beta.

New follow-up estimated spending/reservations: **$0.9762**. Known current-study total including the four earlier runs: **$1.3483**. These figures include a conservative Luna cache-write allowance and are not the account balance.

In [ ]:
from pathlib import Path
import sys, json, os, getpass
import pandas as pd
from IPython.display import display, HTML, Image
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for parent in candidates for p in (parent, parent / "Project/beer_game_experiment")
             if (p / "scripts/run_budget_followup.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the beer_game_experiment project.")
sys.path.insert(0, str(ROOT / "scripts"))
import run_budget_followup as campaign
from budget_model_screen import BudgetStop
SCREEN_DIR = ROOT / "data/development/model_screen_20261005"
REPORT_DIR = ROOT / "reports/budget_followup_20261005"
print("Project:", ROOT)

## 1. Spending and progress — safe to run repeatedly

Costs are calculated/reserved from saved requests, not an account balance. The earlier four current-study runs cost approximately $0.3720. The planned stage limits sum to at most $1.50 for the selected low-cost model, including the separate $0.03 calibration-metadata recheck, leaving a margin under the approximately $3 total budget. Luna input charges include a conservative cache-write allowance. The archive contains older experiments outside this subtotal.

In [ ]:
from html import escape
status = campaign.totals()
display(pd.DataFrame(status["stages"]))
print(f"New follow-up estimated charges/reservations: ${status['new_spend_usd']:.4f}")
print(f"Known current-study total: ${status['known_current_study_total_usd']:.4f}")
for row in status["stages"]:
    total = row["done"] + row["left"]
    display(HTML(f"<b>{escape(row['stage'])}: {escape(row['phase'])}</b> "
                 f"<progress value='{row['done']}' max='{max(1,total)}'></progress> "
                 f"{row['done']} done; {row['left']} left"))

## 2. Resume only if needed

Leave `RUN_API = False` to review saved results. Setting it to `True` resumes the frozen follow-up; completed phases are loaded and are not purchased again. The local spending limits cannot be increased here. If a phase stopped because its allowance or calibration failed, inspect the saved diagnosis first.

The key is requested in a hidden input box, used only in memory and excluded from result files.

In [ ]:
RUN_API = False
if not RUN_API:
    print("Review only: no paid API calls.")
else:
    record = campaign.freeze()
    if record.get("status") == "complete":
        print("Already complete. Saved results loaded; no paid API calls.")
    else:
        from openai import AsyncOpenAI
        _api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not _api_key.strip():
            raise ValueError("No API key entered.")
        _client = AsyncOpenAI(api_key=_api_key.strip(), max_retries=0, timeout=90.)
        del _api_key
        try:
            await campaign.run(_client)
        except BudgetStop as exc:
            print("Stopped:", str(exc))
        finally:
            await _client.close()
            del _client
        display(pd.DataFrame(campaign.totals()["stages"]))

## 3. Model comparison and calibration

The comparison includes states selected from the previous failure, so it is a development check rather than an independent test of generalization. “Eligible” does not establish the hypotheses. Every candidate response remains saved.

In [ ]:
if (SCREEN_DIR / "summary.csv").exists():
    display(pd.read_csv(SCREEN_DIR / "summary.csv"))
record = json.loads(campaign.CAMPAIGN.read_text()) if campaign.CAMPAIGN.exists() else {}
print("Selected model:", record.get("selected", {}).get("model", "not selected"))
if record.get("calibration_dir"):
    calibration_dir = Path(record["calibration_dir"])
    if (calibration_dir / "calibration_summary.csv").exists():
        display(pd.read_csv(calibration_dir / "calibration_summary.csv"))
        print(json.loads((calibration_dir / "calibration_recommendation.json").read_text()))

## 4. Game results and report

Costs are game cost units, separate from API dollars. Beta measures fitted supply-line weight. Unusable fits remain visible. With two independent chain pairs this is a pilot; weekly orders are not independent experimental participants.

In [ ]:
if record.get("game_dir"):
    game_dir = Path(record["game_dir"])
    for name in ("group_outcomes", "manipulation_check", "beta_fits", "hypotheses"):
        if (game_dir / f"{name}.csv").exists():
            print(name.replace("_", " ").title())
            display(pd.read_csv(game_dir / f"{name}.csv"))
    for name in ("order_trajectories.png", "bullwhip.png"):
        if (game_dir / name).exists():
            display(Image(filename=str(game_dir / name)))
else:
    print("No game results yet; calibration must pass first.")
for path in (REPORT_DIR / "Budget_Followup_Report.docx", REPORT_DIR / "DETAILS.xlsx"):
    if path.exists():
        print("Report file:", path)
print("Original pilot report:", ROOT / "reports/pilot_02_game_api_8236a9c840b29d36/Pilot_Report.docx")